# Length Stratified Sampling — Post-Process Notebook

**Purpose:** Post-process synthetic dataset untuk balance length distribution.

**Catatan:**
- Ini **BUKAN core pipeline tool** — post-process optional
- Dipakai kalau analisis post-run menunjukkan distribution tidak balanced
- Applied HANYA kalau perlu (empirical decision)

**Reference:**
- Buda et al. 2018 "A systematic study of the class imbalance problem"
- He & Garcia 2009 "Learning from Imbalanced Data"

**Approach:**
- Stratified sample dari synthetic only (NO seed fallback mixing)
- Seed train concat di akhir — providing natural extreme bucket coverage
- No duplicates, clean separation

## 1. Imports & Configuration

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

ROOT = Path('y:/Michh/Python/Projects/MAGenerator')

# ── Config (edit these) ──
LANG = 'jav'
REFERENCE_SET = 'test'  # 'train' or 'test'
TOTAL_PER_LABEL = None  # None = mean of syn label counts; or set int e.g. 500
OUTPUT_SUFFIX = '_balanced'

# Length buckets
BUCKETS = [(0, 9), (10, 19), (20, 29), (30, 39), (40, 49), (50, 200)]

print(f'Config: lang={LANG}, reference={REFERENCE_SET}, suffix={OUTPUT_SUFFIX}')

## 2. Helper Functions

In [ ]:
def tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

def length_bucket(text, buckets=BUCKETS):
    n = len(tokenize(text))
    for i, (lo, hi) in enumerate(buckets):
        if lo <= n <= hi:
            return i
    return len(buckets) - 1

def compute_target_dist(target_df, buckets=BUCKETS):
    """% per bucket per label dari reference dataset."""
    target_dist = {}
    for label in target_df['label'].unique():
        sub = target_df[target_df['label'] == label]
        bucket_counts = np.zeros(len(buckets))
        for text in sub['text'].dropna():
            bucket_counts[length_bucket(text, buckets)] += 1
        bucket_counts /= bucket_counts.sum()
        target_dist[label] = bucket_counts
    return target_dist

def stratified_sample_synthetic(syn_df, target_dist, total_per_label, buckets=BUCKETS, rng=None):
    """Stratified sampling dari synthetic only. No seed fallback — seed_train concat terpisah."""
    if rng is None:
        rng = np.random.default_rng(42)

    sampled_rows = []
    shortfall_log = defaultdict(int)

    for label, target_props in target_dist.items():
        target_counts = (target_props * total_per_label).astype(int)
        sub_syn = syn_df[syn_df['label'] == label].copy()
        sub_syn['_bucket'] = sub_syn['text'].apply(lambda t: length_bucket(t, buckets))

        for b_idx, target_n in enumerate(target_counts):
            if target_n == 0:
                continue
            available = sub_syn[sub_syn['_bucket'] == b_idx]

            if len(available) >= target_n:
                picked = available.sample(n=target_n, random_state=int(rng.integers(0, 1e6)))
                sampled_rows.append(picked.drop(columns=['_bucket']))
            elif len(available) > 0:
                sampled_rows.append(available.drop(columns=['_bucket']))
                shortfall_log[(label, b_idx)] = target_n - len(available)
            else:
                shortfall_log[(label, b_idx)] = target_n

    if sampled_rows:
        result = pd.concat(sampled_rows, ignore_index=True)
    else:
        result = pd.DataFrame(columns=syn_df.columns)

    return result, shortfall_log

print('Helper functions defined')

## 3. Load Data

In [ ]:
seed_train = pd.read_csv(ROOT / f'data/nusax_senti/{LANG}/train.csv')
test = pd.read_csv(ROOT / f'data/nusax_senti/{LANG}/test.csv')
syn = pd.read_csv(ROOT / f'outputs/synthetic/{LANG}/synthetic.csv')

reference_df = seed_train if REFERENCE_SET == 'train' else test

print(f'Seed train: {len(seed_train)} rows  |  dist: {seed_train["label"].value_counts().to_dict()}')
print(f'Test:       {len(test)} rows  |  dist: {test["label"].value_counts().to_dict()}')
print(f'Synthetic:  {len(syn)} rows  |  dist: {syn["label"].value_counts().to_dict()}')
print(f'Reference for target distribution: {REFERENCE_SET} ({len(reference_df)} rows)')

## 4. Compute Target Distribution

In [ ]:
target_dist = compute_target_dist(reference_df)

print(f'Target distribution (% per bucket per label, from {REFERENCE_SET}.csv):')
print(f'{"label":<10}', end='')
for lo, hi in BUCKETS:
    print(f'  {lo:>2}-{hi:>3}', end='')
print()
for label, props in target_dist.items():
    print(f'  {label:<8}', end='')
    for p in props:
        print(f'  {p*100:>5.1f}%', end='')
    print()

## 5. Check Synthetic Current Distribution

In [ ]:
print(f'Synthetic current distribution (% per bucket per label):')
print(f'{"label":<10}', end='')
for lo, hi in BUCKETS:
    print(f'  {lo:>2}-{hi:>3}', end='')
print()
for label in target_dist.keys():
    sub = syn[syn['label']==label]
    bucket_counts = np.zeros(len(BUCKETS))
    for text in sub['text'].dropna():
        bucket_counts[length_bucket(text)] += 1
    if bucket_counts.sum() > 0:
        bucket_counts_pct = bucket_counts / bucket_counts.sum()
    else:
        bucket_counts_pct = bucket_counts
    print(f'  {label:<8}', end='')
    for p in bucket_counts_pct:
        print(f'  {p*100:>5.1f}%', end='')
    print(f'   (n={int(bucket_counts.sum())})')

print(f'\n→ Compare dengan target di atas. Kalau mirip, tidak perlu stratified sampling.')

## 6. Apply Stratified Sampling

In [ ]:
# Decide total per label
if TOTAL_PER_LABEL:
    total_per_label = TOTAL_PER_LABEL
else:
    total_per_label = int(syn['label'].value_counts().mean())

print(f'Target total synthetic per label: {total_per_label}')
print(f'Expected total synthetic (3 labels): {total_per_label * 3} (maximum jika semua bucket fillable)')

sampled, shortfall = stratified_sample_synthetic(syn, target_dist, total_per_label)

print(f'\nActual sampled: {len(sampled)} rows')
print(f'Per-label: {sampled["label"].value_counts().to_dict()}')

if shortfall:
    print(f'\nShortfall (bucket yang tidak fully filled dari synthetic):')
    total_shortfall = 0
    for (lab, b_idx), n in sorted(shortfall.items()):
        lo, hi = BUCKETS[b_idx]
        print(f'  {lab} bucket {lo}-{hi}: {n} samples short')
        total_shortfall += n
    print(f'  TOTAL shortfall: {total_shortfall}')
    print(f'  → Gap akan di-cover oleh seed_train di final concat')
    print(f'    (seed_train natural includes all buckets)')

## 7. Verify Sampled Distribution

In [ ]:
print(f'Final synthetic sampled distribution (% per bucket per label):')
print(f'{"label":<10}', end='')
for lo, hi in BUCKETS:
    print(f'  {lo:>2}-{hi:>3}', end='')
print()
for label in target_dist.keys():
    sub = sampled[sampled['label']==label]
    bucket_counts = np.zeros(len(BUCKETS))
    for text in sub['text'].dropna():
        bucket_counts[length_bucket(text)] += 1
    if bucket_counts.sum() > 0:
        bucket_counts_pct = bucket_counts / bucket_counts.sum()
    else:
        bucket_counts_pct = bucket_counts
    print(f'  {label:<8}', end='')
    for p in bucket_counts_pct:
        print(f'  {p*100:>5.1f}%', end='')
    print(f'   (n={int(bucket_counts.sum())})')

## 8. Save Outputs

In [ ]:
# Save balanced synthetic (no seed mixing)
output_path = ROOT / f'outputs/synthetic/{LANG}/synthetic{OUTPUT_SUFFIX}.csv'
sampled.to_csv(output_path, index=False)
print(f'Saved balanced synthetic: {output_path}  ({len(sampled)} rows)')

# Build augmented train: seed_train + balanced syn (separate sources, no duplicates)
aug = pd.concat([seed_train, sampled], ignore_index=True)
aug_path = ROOT / f'data/nusax_senti/{LANG}/train_syn{OUTPUT_SUFFIX}.csv'
aug.to_csv(aug_path, index=False)
print(f'Augmented train: {aug_path}  ({len(aug)} rows = {len(seed_train)} seed + {len(sampled)} syn)')

## 9. Final Combined Distribution (Seed + Balanced Syn)

Inspect final dataset untuk verify balanced.

In [ ]:
print(f'Final combined (seed_train + balanced_syn) distribution:')
print(f'{"label":<10}', end='')
for lo, hi in BUCKETS:
    print(f'  {lo:>2}-{hi:>3}', end='')
print()
for label in target_dist.keys():
    sub = aug[aug['label']==label]
    bucket_counts = np.zeros(len(BUCKETS))
    for text in sub['text'].dropna():
        bucket_counts[length_bucket(text)] += 1
    if bucket_counts.sum() > 0:
        bucket_counts_pct = bucket_counts / bucket_counts.sum()
    else:
        bucket_counts_pct = bucket_counts
    print(f'  {label:<8}', end='')
    for p in bucket_counts_pct:
        print(f'  {p*100:>5.1f}%', end='')
    print(f'   (n={int(bucket_counts.sum())})')

print(f'\nTotal final dataset: {len(aug)} rows')
print(f'  - Seed train ({len(seed_train)} rows, natural distribution)')
print(f'  - Balanced synthetic ({len(sampled)} rows, matched target)')
print(f'  - NO duplicates (different sources)')

## Notes

**Decision kapan apply:**

Compare distribution synthetic (Cell 5) vs target (Cell 4):
- Kalau **mirip** (misal ratio 0.85+) → TIDAK perlu stratified sampling
- Kalau **beda signifikan** (e.g., 74% synthetic di 20-29 vs 22% test) → Apply

**Output files:**
- `outputs/synthetic/{lang}/synthetic_balanced.csv` — balanced subset (synthetic only)
- `data/nusax_senti/{lang}/train_syn_balanced.csv` — seed + balanced_syn (ready for fine-tune)

**Clean design:**
- Sampled berisi synthetic only (no seed mixing)
- Seed train concat terpisah
- NO duplicates
- Seed provides natural extreme bucket coverage; balanced syn provides medium bucket coverage

**References:**
- Buda et al. 2018 "A systematic study of the class imbalance problem in convolutional neural networks"
- He & Garcia 2009 "Learning from Imbalanced Data"